In [0]:
pip install ucimlrepo

In [0]:
dbutils.library.restartPython()

In [0]:
# Fetch dataset from UCI repository
from ucimlrepo import fetch_ucirepo
online_retail = fetch_ucirepo ( id =352)
# Get the data as a pandas DataFrame
pandas_df = online_retail . data . features
# Convert pandas to Spark DataFrame
raw_df = spark . createDataFrame ( pandas_df )
raw_df . printSchema () # Look at the structure
display ( raw_df . limit (10) ) # See first 10 rows

In [0]:
from pyspark.sql.functions import col, to_timestamp

# Parse datetime, remove rows with null CustomerID
clean_df = raw_df \
    .withColumn("InvoiceDate", to_timestamp(col("InvoiceDate"), "M/d/yyyy H:mm")) \
    .filter(col("CustomerID").isNotNull())

# Standardize column names (lowercase with underscores)
clean_df = clean_df \
    .withColumnRenamed("CustomerID", "customer_id") \
    .withColumnRenamed("UnitPrice", "unit_price") \
    .withColumnRenamed("Quantity", "quantity")

display(clean_df.limit(10))

In [0]:
# Save as a managed Delta table (Databricks handles storage)
clean_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecommerce_delta_table")

# Verify it worked
display(spark.sql("SELECT COUNT(*) as row_count FROM ecommerce_delta_table"))

In [0]:
%sql
SELECT
    CAST(CAST(customer_id AS INT) AS STRING) AS customer_id,
    SUM(quantity) AS total_items_purchased,
    SUM(unit_price * quantity) AS total_spent
FROM ecommerce_delta_table
WHERE customer_id IS NOT NULL AND quantity > 0
GROUP BY customer_id
ORDER BY total_spent DESC
LIMIT 10;

In [0]:
%sql
DESCRIBE HISTORY ecommerce_delta_table;  -- See all versions

SELECT * FROM ecommerce_delta_table VERSION AS OF 0;  -- Original
SELECT * FROM ecommerce_delta_table VERSION AS OF 1;  -- Version 1

-- Or go back by time
SELECT * FROM ecommerce_delta_table 
  TIMESTAMP AS OF current_timestamp() - INTERVAL 1 HOUR;

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8378872147571707>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'DESCRIBE HISTORY ecommerce_delta_table;  -- See all versions\n\nSELECT * FROM ecommerce_delta_table VERSION AS OF 0;  -- Original\nSELECT * FROM ecommerce_delta_table VERSION AS OF 1;  -- Version 1\n\n-- Or go back by time\nSELECT * FROM ecommerce_delta_table \n  TIMESTAMP AS OF current_timestamp() - INTERVAL 1 HOUR;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python to

In [0]:
%sql
-- Cell 1: Check current versions
DESCRIBE HISTORY ecommerce_delta_table;

In [0]:
%sql
-- Cell 2: Make a change to create a new version
DELETE FROM ecommerce_delta_table WHERE quantity < 0;

In [0]:
%sql
-- Cell 3: Check history again - now you have version 0 AND 1!
DESCRIBE HISTORY ecommerce_delta_table;

In [0]:
%sql
-- Cell 4: Query the ORIGINAL data (before the delete)
SELECT COUNT(*) FROM ecommerce_delta_table VERSION AS OF 0;